# RealPDE Eval on Kaggle (`local_eval.py`)

Editor-first flow: implement variants locally under `submissions/submission_vN/`, `git push`, then here just `git pull` and run. This notebook never writes `submission.py` — it only pulls the repo and calls `local_eval.py --submission submissions/<variant>`.

1. Pull repo → 2. resolve `--data` → 3. smoke-test each variant on `example_data/` → 4. (optional) score on real `test_real` + stage a baseline `model.pth` → 5. pack Codabench zips.

In [ ]:
# --- Clone or pull repo (public, no token) ---
REPO_DIR = '/kaggle/working/realpde'
REPO_URL = 'https://github.com/nthday-jpg/realpde.git'

!if [ -d {REPO_DIR} ]; then echo "Pulling {REPO_DIR}..."; cd {REPO_DIR} && git pull; else echo "Cloning {REPO_URL}..."; git clone {REPO_URL} {REPO_DIR}; fi

In [ ]:
%cd {REPO_DIR}
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
import torch
print(f'torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# --- Resolve --data (example_data fallback, real test_real if attached) ---
from pathlib import Path

REPO = Path(REPO_DIR)
example_data = REPO / 'example_data'
# Kaggle dataset root (cf. continue_cno_kaggle.ipynb): {baseline,test,train_real,train_sim}/ underneath
DATA_ROOT = next((c for c in ['/kaggle/input/datasets/nthday/realpde', '/kaggle/input/realpde'] if Path(c).exists()), None)
print(f'DATA_ROOT -> {DATA_ROOT}')
DATA_DIR = example_data  # default: bundled synthetic smoke test
for cand in ([Path(DATA_ROOT)] if DATA_ROOT else []) + [Path('/kaggle/input/realpde')]:
    if (cand / 'test_real').is_dir() and (cand / 'mean_std_real.pt').exists():
        DATA_DIR = cand
        break
print(f'DATA_DIR -> {DATA_DIR}')
print(f'  example_data present: {(example_data / "test_real").is_dir()}')

# --- List submission variants ---
variants = sorted(p.name for p in (REPO / 'submissions').glob('submission_*') if p.is_dir())
print(f'variants: {variants}')

In [ ]:
# --- Smoke test every variant on example_data (CPU, always works) ---
import subprocess, sys
from pathlib import Path

REPO = Path(REPO_DIR)
for v in sorted(p.name for p in (REPO / 'submissions').glob('submission_*') if p.is_dir()):
    print(f'\n===== {v} (example_data) =====')
    r = subprocess.run([sys.executable, 'local_eval.py', '--submission', f'submissions/{v}',
                        '--data', './example_data'], cwd=REPO)
    print(f'[{v}] exit code: {r.returncode}')

## Real-data check: 30 random trajectories, CNO, no adaptation (GPU)

Stages 30 seeded trajectories from the attached real dataset into `/kaggle/working/real30/` (truncated copies + protocol-exact stats, via `scripts/stage_real30.py`), then scores `submission_v3` + CNO on GPU. Local CPU would take ~hours — here it's minutes. Skips cleanly if no real dataset is attached.

In [ ]:
# --- Stage 30 real trajectories (seeded, reproducible) ---
# Tune these two lines if you want more coverage (GPU-cheap for v3, pricey for v2).
N_FILES, N_FRAMES, SEED = 30, 200, 42
import subprocess, sys
from pathlib import Path

REPO = Path(REPO_DIR)
DATA_ROOT = next((c for c in ['/kaggle/input/datasets/nthday/realpde', '/kaggle/input/realpde'] if Path(c).exists()), None)
train_real = None
if DATA_ROOT is not None:
    base = Path(DATA_ROOT) / 'train_real'
    if sorted(base.glob('*.h5')):
        train_real = base
    elif sorted(base.rglob('*.h5')):  # nested layout: most common parent wins
        from collections import Counter
        train_real = Counter(p.parent for p in base.rglob('*.h5')).most_common(1)[0][0]
print(f'train_real -> {train_real}')
if train_real is None:
    print('[skip] no real trajectories attached — attach the dataset with train_real/')
else:
    r = subprocess.run([sys.executable, 'scripts/stage_real30.py', '--src', str(train_real),
                        '--dst', '/kaggle/working/real30', '--n-files', str(N_FILES),
                        '--frames', str(N_FRAMES), '--seed', str(SEED)], cwd=REPO)
    print(f'[stage] exit code: {r.returncode}')

In [ ]:
# --- Score a variant + CNO on the staged real-30 set (GPU when available) ---
# Set VARIANT to the submission folder under test (v4 = calibrated, v3 = plain).
VARIANT = 'submission_v4'
import shutil, subprocess, sys, torch
from pathlib import Path

REPO = Path(REPO_DIR)
real30 = Path('/kaggle/working/real30')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')
if not (real30 / 'test_real').is_dir():
    print('[skip] /kaggle/working/real30 not staged — run the cell above first')
else:
    roots = [Path('/kaggle/input/datasets/nthday/realpde'), Path('/kaggle/input/realpde'), Path('/kaggle/working/checkpoints')]
    ckpts = [p for r in roots if r.exists() for ext in ('*.pth', '*.pt') for p in r.rglob(ext)]
    cands = sorted([p for p in ckpts if 'cno' in p.name.lower()],
                 key=lambda p: (0 if 'sim_real' in p.name.lower() else 1, p.name))
    ckpt = cands[0] if cands else (sorted(ckpts)[0] if ckpts else None)
    print(f'checkpoint: {ckpt}')
    if ckpt is None:
        print('[skip] no checkpoint found')
    else:
        dst = REPO / 'submissions' / VARIANT / 'model.pth'
        shutil.copy(ckpt, dst)
        print(f'[stage] {ckpt.name} -> submissions/{VARIANT}/model.pth')
        r = subprocess.run([sys.executable, 'local_eval.py', '--submission',
                            f'submissions/{VARIANT}', '--data', str(real30),
                            '--device', device], cwd=REPO)
        print(f'[{VARIANT} real30] exit code: {r.returncode}')
        print('Copy the subscores above into submissions/README.md Leaderboard.')

In [ ]:
# --- Pack Codabench zips (submission.py at root, shared files injected) ---
import subprocess, sys
from pathlib import Path

REPO = Path(REPO_DIR)
for v in sorted(p.name for p in (REPO / 'submissions').glob('submission_*') if p.is_dir()):
    print(f'\n===== packing {v} =====')
    r = subprocess.run([sys.executable, 'scripts/make_submission_zip.py', v], cwd=REPO)
    print(f'[{v}] pack exit code: {r.returncode}')
print('\nZips:')
!ls -la dist/ 2>/dev/null || echo '(no dist/ yet — pack a variant first)'